In [ ]:
#@title Lingua dual-model trainer — self-resuming (run this ONE cell; re-run after any disconnect)
# Checkpoints live on Drive: a disconnect loses NOTHING. The while-loop restarts
# training after any crash; the supervising web session fixes pushed errors hourly.
from google.colab import drive
drive.mount('/content/drive')

%cd /content
![ -d Lingua-Sound-Wave ] || git clone https://github.com/grundlagen/Lingua-Sound-Wave
%cd Lingua-Sound-Wave
!git fetch origin && git checkout claude/phrase-weave-multiword && git pull --rebase origin claude/phrase-weave-multiword
!pip -q install transformers trl datasets accelerate sentencepiece panphon wordfreq sentence_transformers
!apt-get -qq install -y espeak-ng > /dev/null

%cd research/homophone-bench
![ -f train-dual-v1.jsonl ] || python build_train_corpus.py

CKPT = "/content/drive/MyDrive/lingua-ckpt"
import subprocess, time, traceback
while True:
    # pick up hourly code fixes from the supervising session
    subprocess.run("git pull --rebase origin claude/phrase-weave-multiword", shell=True)
    r = subprocess.run(
        f"python selflearn/train_selflearn.py --data train-dual-v1.jsonl "
        f"--continual --ckpt_dir {CKPT} --rounds 1",
        shell=True)
    if r.returncode != 0:
        # push the traceback so the supervisor can fix the code
        subprocess.run("git add TRAIN_ERRORS.log 2>/dev/null; "
                       "git commit -qm 'colab: TRAIN_ERRORS traceback' 2>/dev/null; "
                       "git pull --rebase -q origin claude/phrase-weave-multiword; "
                       "git push -q origin claude/phrase-weave-multiword", shell=True)
        print("crash pushed to TRAIN_ERRORS.log; retrying in 5 min (supervisor fixes hourly)")
        time.sleep(300)
